# Assignment 1: E-Commerce Data Cleaning Pipeline

Is notebook mein hum e-commerce ke raw JSON data ko step-by-step clean aur structure karenge:
1. **Data Loading**: JSON file se Python data structure (`list` of `dict`) mein load karna
2. **Rating Sanitize**: String numbers (`"5"`, `"3.5"`) aur word numbers (`"four"`, `"two"`) ko standard float mein convert karna
3. **String & Space Cleaning**: Names aur feedback se unwanted extra spaces hatana aur standard casing karna
4. **Missing Values**: Missing fields (jaise Rahul ki age missing hai) ko safely `None` assign karna
5. **Deduplication**: Case-insensitive duplicate users ko detect karke filter karna

## Step 1: Raw Data Ko Load Karna (JSON -> Python)
- **Problem**: Server se aane wala data raw JSON text format mein hota hai.
- **Process**: `json.load()` use karke isko Python ke `list` of `dict` format mein convert karte hain taaki aage processing ho sake.

In [19]:
import json

def load_data(file_path):
    with open(file_path, "r") as file:
        return json.load(file)

# store_data.json se raw data load karte hain
raw_data = load_data("store_data.json")
print(f"Total raw records: {len(raw_data)}")
print("\nRaw Data:")
for item in raw_data:
    print(item)

Total raw records: 6

Raw Data:
{'name': 'Alice', 'ratng': '5', 'feedback': 'great product!!', 'age': '25'}
{'name': 'bob', 'ratng': 'four', 'feedback': 'ok but late delivery', 'age': '30'}
{'name': 'diana', 'ratng': '5', 'feedback': 'love it', 'age': '28'}
{'name': 'rahul', 'ratng': 'two', 'feedback': 'bad experiance'}
{'name': 'Eve', 'ratng': '3.5', 'feedback': 'average, not bad', 'age': '35'}
{'name': 'alice', 'ratng': '5', 'feedback': 'great product', 'age': '25'}


## Step 2: Rating Ko Clean & Sanitize Karna
- **Problem**:
  1. Key mein typo hai (`"ratng"` vs `"rating"`).
  2. Ratings inconsistent format mein hain: Kuch words mein (`"four"`, `"two"`), kuch string number mein (`"5"`, `"3.5"`).
- **System Design Solution**: Ek Lookup Dictionary (Word-to-Number mapping) banayenge, aur har rating ko standard `float` type banayenge.

In [12]:
# Words ko numbers mein map karne ke liye lookup table
word_to_num = {
    "one": 1.0,
    "two": 2.0,
    "three": 3.0,
    "four": 4.0,
    "five": 5.0
}

def clean_rating(raw_val):
    if raw_val is None:
        return None
    
    val_str = str(raw_val).strip().lower()
    
    # Agar rating word mein likhi ho (e.g. 'four' -> 4.0)
    if val_str in word_to_num:
        return word_to_num[val_str]
    
    # Agar number string ho (e.g. '5' -> 5.0 ya '3.5' -> 3.5)
    try:
        return float(val_str)
    except ValueError:
        return None

# Sample test run
print("Sample conversions:")
for test in ["5", "four", "3.5", "two"]:
    res = clean_rating(test)
    print(f"Raw: {test:<6} -> Cleaned: {res} (type: {type(res).__name__})")

Sample conversions:
Raw: 5      -> Cleaned: 5.0 (type: float)
Raw: four   -> Cleaned: 4.0 (type: float)
Raw: 3.5    -> Cleaned: 3.5 (type: float)
Raw: two    -> Cleaned: 2.0 (type: float)


## Step 3 & 4: Spaces, Casing & Missing Data (Age)
- **Spaces**: Strings se aage-peeche ki extra spaces `.strip()` se hatayenge, aur names ko `.title()` ya `.capitalize()` karenge.
- **Missing Data**: Rahul ke record mein `age` key hi nahi hai. `.get("age", None)` use karke missing data ko safely `None` set karenge taaki `KeyError` na aaye.

In [13]:
def clean_single_record(item):
    # 1. Name clean (strip spaces, capitalize first letter)
    raw_name = item.get("name", "")
    name = raw_name.strip().capitalize() if raw_name else "Unknown"
    
    # 2. Rating clean (handle key typo 'ratng' or 'rating')
    raw_rating = item.get("ratng") if "ratng" in item else item.get("rating")
    rating = clean_rating(raw_rating)
    
    # 3. Feedback clean
    feedback = item.get("feedback", "").strip()
    
    # 4. Age clean (handle missing safely as None)
    raw_age = item.get("age")
    age = int(raw_age) if (raw_age is not None and str(raw_age).strip().isdigit()) else None
    
    return {
        "name": name,
        "rating": rating,
        "feedback": feedback,
        "age": age
    }

## Step 5: Duplicate Data Clean Karna (Deduplication)
- **Problem**: Raw data mein `"Alice"` (age 25) aur `"alice"` (age 25) do entries hain.
- **Process**: Ek `seen_users = set()` maintain karenge. Har record ka unique signature `(name.lower(), age)` check karenge. Agar pehle se dekha hua hai, toh skip kar denge.

In [8]:
def clean_ecommerce_pipeline(raw_data):
    cleaned_data = []
    seen_users = set()
    
    for item in raw_data:
        cleaned_item = clean_single_record(item)
        
        # Unique identifier banaya (case-insensitive name + age)
        unique_key = (cleaned_item["name"].lower(), cleaned_item["age"])
        
        if unique_key in seen_users:
            print(f"--> [Duplicate Removed]: {cleaned_item['name']} (Age: {cleaned_item['age']})")
            continue
            
        seen_users.add(unique_key)
        cleaned_data.append(cleaned_item)
        
    return cleaned_data

# Pipeline run karte hain
cleaned_dataset = clean_ecommerce_pipeline(raw_data)

--> [Duplicate Removed]: Alice (Age: 25)


## Final Output: Cleaned Structured Data
Ab check karte hain hamara clean data kaisa dikhta hai aur kya saare 5 rules verify ho gaye!

In [17]:
print("=" * 60)
print(f"Raw Count: {len(raw_data)} | Cleaned Count: {len(cleaned_dataset)}")
print("=" * 60)

for idx, item in enumerate(cleaned_dataset, 1):
    print(f"{idx}. Name: {item['name']:<8} | Rating: {item['rating']} | Age: {str(item['age']):<4} | Feedback: {item['feedback']}")

Raw Count: 6 | Cleaned Count: 5
1. Name: Alice    | Rating: 5.0 | Age: 25   | Feedback: great product!!
2. Name: Bob      | Rating: 4.0 | Age: 30   | Feedback: ok but late delivery
3. Name: Diana    | Rating: 5.0 | Age: 28   | Feedback: love it
4. Name: Rahul    | Rating: 2.0 | Age: None | Feedback: bad experiance
5. Name: Eve      | Rating: 3.5 | Age: 35   | Feedback: average, not bad
